
<font color = green >

# Home Task

</font>

Objective: Apply at least two (2) modern sentiment analysis methods to classify text data and evaluate their performance




Dataset:

Sentiment Analysis Dataset: https://www.cs.cornell.edu/people/pabo/movie-review-data/rt-polaritydata.tar.gz

alternative source:
[rt-polaritydata](https://github.com/dennybritz/cnn-text-classification-tf/tree/master/data/rt-polaritydata)

Each line in these two files corresponds to a single snippet (usually containing roughly one single sentence); all snippets are down-cased.

[More info about dataset](https://www.cs.cornell.edu/people/pabo/movie-review-data/rt-polaritydata.README.1.0.txt)


- rt-polarity.neg: Contains negative reviews.
- rt-polarity.pos: Contains positive reviews

## Task Description:

1. Data Loading & Preparation:

    - Load rt-polarity.neg and rt-polarity.pos.
    - Split each file into individual snippets.
    - Assign labels (0 for negative, 1 for positive).
    - Combine into a single dataset.
    - Split the dataset into training and testing sets.
2. Implement and evaluate at least three (3) methods from the lecture, prioritizing modern approaches.
3. For each implemented method, report and compare classification metrics: Accuracy, Precision, Recall, and F1-score.

### 1. Data Loading & Preparation:

####  Load rt-polarity.neg and rt-polarity.pos.
#### Split each file into individual snippets.

In [1]:
fn='rt-polarity.neg'

with open(fn, "r",encoding='utf-8', errors='ignore') as f: # some invalid symbols encountered
    content = f.read()
texts_neg=  content.splitlines()
print ('len of texts_neg = {:,}'.format (len(texts_neg)))
for review in texts_neg[:5]:
    print ( '\n', review)

len of texts_neg = 5,331

 simplistic , silly and tedious . 

 it's so laddish and juvenile , only teenage boys could possibly find it funny . 

 exploitative and largely devoid of the depth or sophistication that would make watching such a graphic treatment of the crimes bearable . 

 [garbus] discards the potential for pathological study , exhuming instead , the skewed melodrama of the circumstantial situation . 

 a visually flashy but narratively opaque and emotionally vapid exercise in style and mystification . 


In [2]:
fn='rt-polarity.pos'

with open(fn, "r",encoding='utf-8', errors='ignore') as f:
    content = f.read()
texts_pos=  content.splitlines()
print ('len of texts_pos = {:,}'.format (len(texts_pos)))
for review in texts_pos[:5]:
    print ('\n', review)

len of texts_pos = 5,331

 the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal . 

 the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson's expanded vision of j . r . r . tolkien's middle-earth . 

 effective but too-tepid biopic

 if you sometimes like to go to the movies to have fun , wasabi is a good place to start . 

 emerges as something rare , an issue movie that's so honest and keenly observed that it doesn't feel like one . 


#### Assign labels (0 for negative, 1 for positive).

In [3]:
y_neg = [0] * len(texts_neg)
y_pos = [1] * len(texts_pos)

#### Combine into a single dataset.

In [4]:
X = texts_neg + texts_pos
y = y_neg + y_pos

In [5]:
len(X)

10662

#### Split the dataset into training and testing sets.

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state=42,
    stratify=y
)

In [7]:
len(X_train)

7996

### 2. Implement and evaluate at least three (3) methods from the lecture, prioritizing modern approaches.

#### 1. TF-IDF + LogisticRegression

In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import numpy as np

In [33]:
tfidf_vectorizer = TfidfVectorizer(min_df=2, stop_words='english').fit(X_train)
X_train_vectorized= tfidf_vectorizer.transform(X_train)
print (list(tfidf_vectorizer.vocabulary_.items())[:5])

[('gussied', 3247), ('distracting', 2056), ('special', 6683), ('effects', 2274), ('visual', 7805)]


In [36]:
sorted_tfidf_index = X_train_vectorized.max(axis=0).toarray()[0].argsort()
print (np.sort(X_train_vectorized.max(axis=0).toarray()[0]))
sorted_tfidf_index

[0.24691335 0.25485948 0.25485948 ... 1.         1.         1.        ]


array([3885, 7138, 3256, ..., 5797, 4526, 8009], dtype=int64)

In [37]:
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
print ('feature_names ',feature_names)
print('Smallest tfidf:\n{}\n'.format(feature_names[sorted_tfidf_index[:10]]))
print('Largest tfidf: \n{}'.format(feature_names[sorted_tfidf_index[:-11:-1]]))

feature_names  ['007' '10' '100' ... 'zings' 'zombie' 'zone']
Smallest tfidf:
['jeopardy' 'talento' 'hace' 'pues' 'tapping' 'pero' 'larry' 'dani'
 'alfonso' 'amir']

Largest tfidf: 
['wise' 'mess' 'refreshing' 'silly' 'funny' 'tiring' 'terrible' 'gut'
 'say' 'clever']


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [38]:
clf = LogisticRegression(max_iter=1000).fit(X_train_vectorized, y_train) # Train the model
predictions = clf.predict(tfidf_vectorizer.transform(X_test))

In [39]:
precision_tfidf = precision_score(y_test, predictions)
recall_tfidf = recall_score(y_test, predictions)
f1_tfidf = f1_score(y_test, predictions)
accuracy_tfidf = accuracy_score(y_test, predictions)

#### 2. Ready-made transformer pipeline

In [ ]:
from transformers import pipeline
import torch

In [45]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [49]:
sample_reviews = X_train[:5]  
# Perform emotion analysis
for review in sample_reviews:
    # Truncate long reviews for demonstration
    result = sentiment_pipeline(review[:512])
    print(f"Review: {review[:100]}...")
    print(f"Predicted sentiment: {result[0]['label']} ({result[0]['score']:.4f})")
    print("-" * 50)

Review: gussied up with so many distracting special effects and visual party tricks that it's not clear whet...
Predicted sentiment: NEGATIVE (0.9997)
--------------------------------------------------
Review: shiri is an action film that delivers on the promise of excitement , but it also has a strong dramat...
Predicted sentiment: POSITIVE (0.9997)
--------------------------------------------------
Review: fails to bring as much to the table . ...
Predicted sentiment: NEGATIVE (0.9997)
--------------------------------------------------
Review: it might be the first sci-fi comedy that could benefit from a three's company-style laugh track . ...
Predicted sentiment: POSITIVE (0.9953)
--------------------------------------------------
Review: i stopped thinking about how good it all was , and started doing nothing but reacting to it - feelin...
Predicted sentiment: POSITIVE (0.9997)
--------------------------------------------------


In [50]:
y_pred = []

for text in X_test:
    result = sentiment_pipeline(text[:512])[0]  # truncate safety
    label = result["label"]

    if label == "POSITIVE":
        y_pred.append(1)
    else:
        y_pred.append(0)

In [51]:
precision_rmt = precision_score(y_test, y_pred)
recall_rmt = recall_score(y_test, y_pred)
f1_rmt = f1_score(y_test, y_pred)
accuracy_rmt = accuracy_score(y_test, y_pred)

#### 3. Fine-tuned transformer

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
import torch.nn.functional as F
from datasets import Dataset

In [ ]:
X_train_small = X_train[:5000]
y_train_small = y_train[:5000]

train_dataset = Dataset.from_dict({
    'text': X_train_small,
    'label': y_train_small
})

In [ ]:
X_test_small = X_test[:1250]
y_test_small = y_test[:1250]

test_dataset = Dataset.from_dict({
    'text': X_test_small,
    'label': y_test_small
})


In [ ]:
model_name = "distilbert-base-uncased"  
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
# Tokenize data
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1250 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [18]:
# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

# Train the model
trainer.train()

c:\Users\redmi\anaconda3\envs\py310\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,0.694468
20,0.685668
30,0.693443
40,0.671544
50,0.663812
60,0.610694
70,0.471574
80,0.414013
90,0.466172
100,0.399242


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\redmi\anaconda3\envs\py310\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=939, training_loss=0.2663004466583427, metrics={'train_runtime': 4397.2986, 'train_samples_per_second': 3.411, 'train_steps_per_second': 0.214, 'total_flos': 496752744960000.0, 'train_loss': 0.2663004466583427, 'epoch': 3.0})

In [41]:
predictions = trainer.predict(tokenized_test)


c:\Users\redmi\anaconda3\envs\py310\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [42]:
preds = np.argmax(predictions.predictions, axis=-1)

precision_ftt = precision_score(y_test_small, preds)
recall_ftt = recall_score(y_test_small, preds)
f1_ftt = f1_score(y_test_small, preds)
accuracy_ftt = accuracy_score(y_test_small, preds)

### 3. For each implemented method, report and compare classification metrics: Accuracy, Precision, Recall, and F1-score.

In [52]:
print("TF-IDF + Logistic Regression model results:")
print(f"Precision: {precision_tfidf:.4f}")
print(f"Recall: {recall_tfidf:.4f}")
print(f"F1 Score: {f1_tfidf:.4f}")
print(f"Accuracy: {accuracy_tfidf:.4f}\n")

print("Ready-made transformer model results:")
print(f"Precision: {precision_rmt:.4f}")
print(f"Recall: {recall_rmt:.4f}")
print(f"F1 Score: {f1_rmt:.4f}")
print(f"Accuracy: {accuracy_rmt:.4f}\n")

print(f"Fine-tuned transformer (DistilBERT) model results:")
print(f"Precision: {precision_ftt:.4f}")
print(f"Recall: {recall_ftt:.4f}")
print(f"F1 Score: {f1_ftt:.4f}")
print(f"Accuracy: {accuracy_ftt:.4f}")

TF-IDF + Logistic Regression model results:
Precision: 0.7537
Recall: 0.7599
F1 Score: 0.7568
Accuracy: 0.7558

Ready-made transformer model results:
Precision: 0.8793
Recall: 0.8965
F1 Score: 0.8878
Accuracy: 0.8867

Fine-tuned transformer (DistilBERT) model results:
Precision: 0.8215
Recall: 0.8641
F1 Score: 0.8423
Accuracy: 0.8400
